In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# How to Compute the Conjunctive Normal Form

Formulas are represented as nested tuples.  In order to convert a string into a nested tuple we use the *parser* that is found in the notebook `Propositional-Logic-Parser.ipynb`.

In [ ]:
import { LogicParser } from './PropositionalLogicParser';
import { RecursiveSet } from './Recursive-Set';

In [ ]:
type Variable = string;
type Formula = string | [string, ...Formula[]];
type Literal = Variable | ['¬', Variable];
type Clause = RecursiveSet<Literal>;
type CNF = RecursiveSet<Clause>;

In [ ]:
function parse(s: string): Formula {
    const parser = new LogicParser(s);
    return parser.parse();
}

The function `eliminateBiconditional(f)` takes a formula `f` from propositional logic and eliminates all occurrences of the operator '↔' from this formula.  This is done by using the following equivalence:
$$ (g \leftrightarrow h) \;\Leftrightarrow\; (g \rightarrow h) \wedge (h \rightarrow g) $$

In [ ]:
function eliminateBiconditional(f: Formula): Formula | null {
    'Eliminate the logical operator "↔" from f.'
    // This case covers variables.
    if (typeof f === 'string') {
        return f;
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        switch (op) {
            case '↔': {
                const [g, h] = args as [Formula, Formula];
                return eliminateBiconditional(['∧', ['→', g, h], ['→', h, g]]);
            }
            case '⊤':
            case '⊥':
                return f;
            
            case '¬': {
                const [g] = args as [Formula];
                return ['¬', eliminateBiconditional(g)];
            }
            case '→':
            case '∧':
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                return [op, eliminateBiconditional(g), eliminateBiconditional(h)];
            }
        }
    }
    return null;
}

The function $\texttt{eliminateConditional}(f)$ takes a formula $f$ from propositional logic and eliminates all occurrences of the operator '→' from this formula.  This is done by using the following equivalence:
$$ (g \rightarrow h) \;\Leftrightarrow\; (\neg g \vee h) $$

In [ ]:
function eliminateConditional(f: Formula): Formula | null {
    'Eliminate the logical operator "→" from f.'
    // variables.
    if (typeof f === 'string') { 
        return f; 
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        switch (op) {
            case '⊤':
            case '⊥':
                return f;
            case '→': {
                const [g, h] = args as [Formula, Formula];
                return eliminateConditional(['∨', ['¬', g], h]);
            }
            case '¬': {
                const [g] = args as [Formula];
                return ['¬', eliminateConditional(g)!];
            }
            case '∧':
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                return [op, eliminateConditional(g)!, eliminateConditional(h)!];
            }
        }
    }
    return null;
}

The function $\texttt{nnf}(f)$ computes the *negation normal form* of $f$, while $\texttt{neg}(f)$ computes the *negation normal form* of $\neg f$.  The expression $\texttt{nnf}(f)$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
</ol>
The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
     <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2). 
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
</ol>

The forward declaration for the function `neg` is needed to typecheck the function `nnf`.

In [ ]:
function nnf(f: Formula): Formula | null {
    "Compute the negation normal form of f."
    if (typeof f === 'string') {
        return f;
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        switch (op) {
            case '⊤':
            case '⊥':
                return f;
            case '¬': {
                const [g] = args as [Formula];
                return neg(g);
            }
            case '∧':
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                return [op, nnf(g)!, nnf(h)!];
            }
        }
    }
    return null;
}

function neg(f: Formula): Formula | null {
    'Compute the negation normal form of ¬f.'
    if (typeof f === 'string') {
        return ['¬', f];
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        switch (op) {
            case '⊤':
                return ['⊥'];
            case '⊥':
                return ['⊤'];
            case '¬': {
                const [g] = args as [Formula];
                return nnf(g);
            }
            case '∧': {
                const [g, h] = args as [Formula, Formula];
                return ['∨', neg(g)!, neg(h)!];
            }
            
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                return ['∧', neg(g)!, neg(h)!];
            }
        }
    }
    return null;
}

The function $\texttt{cnf}(f)$ takes a formula $f$ that is in *negation normal form*, i.e. the negation operator is only applied to propositional variables and returns the *conjunctive normal form* of $f$ in *set notation*.  In order to achieve
this it uses the distributive law
$$ (f \wedge g) \vee (h \wedge k) \Leftrightarrow (f \vee h) \wedge (f \vee k) \wedge (g \vee h) \wedge (g \vee k). $$

In [ ]:
function cnf(f: Formula): CNF | null {
    // f is a variable
    if (typeof f === 'string') { 
        const lit = f as Literal;
        const clause = new RecursiveSet<Literal>(lit);
        return new RecursiveSet<Clause>(clause);
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        switch (op) {
            case '⊤':
                return new RecursiveSet<Clause>(); 
            case '⊥':
                const emptyClause = new RecursiveSet<Literal>();
                return new RecursiveSet<Clause>(emptyClause);

            case '¬': {
                const [p] = args as [string];
                const lit: Literal = ['¬', p]; 
                const clause = new RecursiveSet<Literal>(lit);
                return new RecursiveSet<Clause>(clause);
            }

            case '∧': {
                const [g, h] = args as [Formula, Formula];
                const left = cnf(g)!;
                const right = cnf(h)!;
                return left.union(right);
            }

            case '∨': {
                const [g, h] = args as [Formula, Formula];
                const left = cnf(g)!;
                const right = cnf(h)!;
                const result = new RecursiveSet<Clause>();
                for (const c1 of left) {
                    for (const c2 of right) {
                        const unionClause = (c1 as Clause).union(c2 as Clause);
                        result.add(unionClause);
                    }
                }
                return result;
            }
        }
    }
    return null;
}

The function $\texttt{isTrivial}(C)$ checks whether the clause $C$ is *trivial*.

In [ ]:
function isTrivial(clause: Clause): boolean {
    for (const lit of clause) {
        const comp = getComplement(lit as Literal);
        if (clause.has(comp)) {
            return true;
        }
    }
    return false;
}

function getComplement(l: Literal): Literal {
    if (Array.isArray(l)) {
        return l[1];
    } else {
        return ['¬', l];
    }
}

The function `removeDuplicatesFromClause` is necessary because in TypeScript, sets compare objects (like tuples representing literals) by reference, not by value. This means duplicates of structurally identical literals can exist as different objects in a set. The function converts each literal to a canonical string form, uses this string as a key to detect duplicates, and rebuilds the clause with only unique literals. This ensures clauses in CNF do not contain redundant duplicates.

The function $\texttt{simplify}(Cs)$ takes a set of clauses and removes all trivial clauses from $Cs$.

In [ ]:
function simplify(clauses: CNF): CNF {
    const result = new RecursiveSet<Clause>();
    for (const clause of clauses) {
        if (!isTrivial(clause)) {
            result.add(clause);
        }
    }
    return result;
}

The function $\texttt{normalize}$ takes a propositional formula $f$ and transforms $f$ into *conjunctive normal form*.  
Furthermore, trivial clausues are removed.

In [ ]:
function normalize(f: Formula): CNF {
    const n1 = eliminateBiconditional(f);
    const n2 = eliminateConditional(n1);
    const n3 = nnf(n2);
    const n4 = cnf(n3);
    return simplify(n4);
}

In [ ]:
function prettify(M: CNF): string {
    return M.toString();
}

In [ ]:
function test(s: string): string {
    const f = parse(s);
    console.log(`The knf of ${s} is:`);
    return prettify(normalize(f));
}

In [ ]:
test('(¬p → q) → (p → q) → q');

In [ ]:
test('(a → b) ↔ (¬a ∧ ¬b)');

In [ ]:
test('(p ∧ q → r) ∨ ¬r → ¬p');

In [ ]:
test('⊤');

In [ ]:
test('⊥');

In [ ]:
test('(p ∧ q → r) ∨ ¬r → ¬p ↔ ¬p');

In [ ]:
test("p → q");

In [ ]:
test("(p ∧ q) → r");

In [ ]:
test("p ↔ q");

In [ ]:
test("(p → q) ∧ (q → r)");

In [ ]:
test("¬(p ∨ q)");

In [ ]:
test('p ∧ p');